# 04 — Model Validation

Validate the customer segmentation results using cluster sizes, behavioral differences, feature sanity checks, and data-quality checks.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

df = pd.read_csv("../outputs/customer_clusters.csv")

print("=" * 70)
print("MODEL VALIDATION — CUSTOMER SEGMENTATION")
print("=" * 70)
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
# 1. Cluster distribution
cluster_counts = df["Cluster"].value_counts().sort_index()
cluster_percentages = (cluster_counts / len(df) * 100).round(2)

validation_table = pd.DataFrame({
    "Customers": cluster_counts,
    "Percentage": cluster_percentages
})

print("Cluster distribution:")
print(validation_table)


In [ ]:
# 2. Clustering features
features = [
    "Recency",
    "Frequency",
    "Monetary_Value",
    "Average_Order_Value",
    "Engagement_Score",
    "Discount_Dependency",
    "Return_Rate",
    "Online_Purchase_Ratio",
    "InStore_Purchase_Ratio",
    "Avg_Items_Per_Transaction",
    "Support_Interaction_Rate"
]

missing_features = [col for col in features if col not in df.columns]
if missing_features:
    raise ValueError(f"Missing expected features: {missing_features}")

cluster_means = (
    df.groupby("Cluster")[features]
      .mean()
      .round(3)
)

print("Cluster mean profile:")
print(cluster_means)


In [ ]:
# 3. Compare Cluster 1 against Cluster 0
if 0 in cluster_means.index and 1 in cluster_means.index:
    cluster_0 = cluster_means.loc[0]
    cluster_1 = cluster_means.loc[1]

    comparison = pd.DataFrame({
        "Cluster_0": cluster_0,
        "Cluster_1": cluster_1
    })

    comparison["Difference"] = (
        comparison["Cluster_1"] - comparison["Cluster_0"]
    ).round(3)

    comparison["Difference_%"] = (
        comparison["Difference"]
        / comparison["Cluster_0"].replace(0, np.nan)
        * 100
    ).round(2)

    print("Cluster comparison:")
    print(comparison)
else:
    comparison = pd.DataFrame()
    print("Fewer than two clusters; comparison skipped.")


In [ ]:
# 4. Feature descriptive statistics
print("Feature descriptive statistics:")
print(df[features].describe().T)


In [ ]:
# 5. Check ratio/proportion features carefully
ratio_features = [
    "Online_Purchase_Ratio",
    "InStore_Purchase_Ratio",
    "Discount_Dependency",
    "Return_Rate"
]

print("Ratio feature statistics:")
print(df[ratio_features].describe().T)

print("\nSelected quantiles:")
print(
    df[ratio_features].quantile(
        [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
)


In [ ]:
# 6. Data-quality checks
numeric_df = df[features].apply(pd.to_numeric, errors="coerce")

checks = {
    "Duplicate rows": int(df.duplicated().sum()),
    "Missing feature values": int(numeric_df.isna().sum().sum()),
    "Infinite feature values": int(np.isinf(numeric_df.to_numpy()).sum()),
    "Negative Recency": int((df["Recency"] < 0).sum()),
    "Negative Frequency": int((df["Frequency"] < 0).sum()),
    "Negative Monetary Value": int((df["Monetary_Value"] < 0).sum()),
    "Negative Average Order Value": int((df["Average_Order_Value"] < 0).sum()),
    "Negative Engagement Score": int((df["Engagement_Score"] < 0).sum()),
    "Negative Discount Dependency": int((df["Discount_Dependency"] < 0).sum()),
    "Negative Return Rate": int((df["Return_Rate"] < 0).sum()),
    "Negative Online Purchase Ratio": int((df["Online_Purchase_Ratio"] < 0).sum()),
    "Negative InStore Purchase Ratio": int((df["InStore_Purchase_Ratio"] < 0).sum()),
    "Negative Items per Transaction": int((df["Avg_Items_Per_Transaction"] < 0).sum()),
    "Negative Support Interaction Rate": int((df["Support_Interaction_Rate"] < 0).sum())
}

quality_table = pd.DataFrame.from_dict(
    checks, orient="index", columns=["Count"]
)

print("Data-quality checks:")
print(quality_table)


In [ ]:
# 7. Feature ranges
feature_ranges = pd.DataFrame({
    "Min": df[features].min(),
    "Max": df[features].max(),
    "Range": df[features].max() - df[features].min(),
    "Std": df[features].std()
}).sort_values("Range", ascending=False)

print("Feature ranges:")
print(feature_ranges)


In [ ]:
# 8. Standardized cluster differences
overall_mean = df[features].mean()
overall_std = df[features].std().replace(0, np.nan)

standardized_profile = (
    (cluster_means[features] - overall_mean) / overall_std
).round(2)

print("Standardized cluster differences:")
print(standardized_profile)


In [ ]:
# 9. Save validation outputs
validation_table.to_csv("../outputs/validation_cluster_sizes.csv")
cluster_means.to_csv("../outputs/validation_cluster_means.csv")
quality_table.to_csv("../outputs/data_quality_validation.csv")
feature_ranges.to_csv("../outputs/feature_range_validation.csv")
standardized_profile.to_csv("../outputs/standardized_cluster_profile.csv")

if not comparison.empty:
    comparison.to_csv("../outputs/cluster_validation.csv")

print("=" * 70)
print("VALIDATION OUTPUTS SAVED")
print("=" * 70)
print("../outputs/validation_cluster_sizes.csv")
print("../outputs/validation_cluster_means.csv")
print("../outputs/data_quality_validation.csv")
print("../outputs/feature_range_validation.csv")
print("../outputs/standardized_cluster_profile.csv")
if not comparison.empty:
    print("../outputs/cluster_validation.csv")


## Validation conclusion

Review the outputs before changing K or retraining. In particular, investigate unusually large purchase-ratio or support-interaction values at the source-feature calculation stage. A high silhouette score alone does not prove that the business segmentation is correct.
